# 06 — Extension: Redis Streams as a Message Broker

**Goal:** Understand why an external message broker eliminates the buffering problem entirely, and learn the basics of Redis Streams.

## Why Redis?

The sbot buffering bug exists because producer (agent) and consumer (channel) share ONE event loop in ONE process. The producer must explicitly yield for the consumer to run.

With Redis, they're in SEPARATE processes:
```
Process A (Agent):    emit → XADD to Redis → done, keep working
Process B (Telegram): XREAD from Redis → POST to Telegram API
```

No shared event loop. No yielding. No buffering. They're completely independent.

## Prerequisites

Install Redis and the Python client:
```bash
brew install redis          # macOS
redis-server &              # start Redis
pip install redis[hiredis]  # Python client
```

## Exercise 6.1 — Redis basics: key/value

In [ ]:
import redis

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# Set and get
r.set('greeting', 'hello from sbot')
print(r.get('greeting'))

# Expire after 10 seconds
r.setex('temp', 10, 'I disappear in 10s')
print(r.get('temp'))
print(f"TTL: {r.ttl('temp')}s")

## Exercise 6.2 — Redis Streams: append-only log

A Redis Stream is like a log file that multiple consumers can read from. Perfect for message passing.

- `XADD` = append a message
- `XREAD` = read messages (blocking, like `queue.get()`)
- Each message has an auto-generated ID (timestamp-based)

In [ ]:
# Clean up from previous runs
r.delete('sbot:outbound:telegram')

# Producer: add messages to a stream
r.xadd('sbot:outbound:telegram', {'type': 'thinking', 'text': 'thinking about it...'})
r.xadd('sbot:outbound:telegram', {'type': 'tool_call', 'text': 'read_file(main.py)'})
r.xadd('sbot:outbound:telegram', {'type': 'response', 'text': 'Here is your answer!'})

# Consumer: read all messages
messages = r.xrange('sbot:outbound:telegram')
for msg_id, data in messages:
    print(f"  [{msg_id}] {data['type']}: {data['text']}")

### Question 6.2
Each message has a unique ID like `1710500000000-0`. What's the advantage of this over a simple queue?

*Your answer:*



## Exercise 6.3 — Blocking read (like queue.get())

`XREAD` with `block=0` waits until a new message arrives — just like `await queue.get()`.

In [ ]:
import threading
import time

r.delete('sbot:demo')

def producer():
    """Adds messages with delays (simulates agent)."""
    time.sleep(1)
    r.xadd('sbot:demo', {'text': 'thinking...'})
    print(f"[{time.time():.1f}] Producer: sent 'thinking...'")
    
    time.sleep(2)  # simulates LLM call
    r.xadd('sbot:demo', {'text': 'tool call'})
    print(f"[{time.time():.1f}] Producer: sent 'tool call'")
    
    time.sleep(0.5)
    r.xadd('sbot:demo', {'text': 'response!'})
    print(f"[{time.time():.1f}] Producer: sent 'response!'")

def consumer():
    """Reads messages as they arrive."""
    last_id = '0'  # start from beginning
    for _ in range(3):  # read 3 messages
        result = r.xread({'sbot:demo': last_id}, block=5000, count=1)
        if result:
            stream, messages = result[0]
            for msg_id, data in messages:
                print(f"[{time.time():.1f}]   Consumer: got '{data['text']}'")
                last_id = msg_id

# Run in separate threads (simulating separate processes)
t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)
t2.start(); t1.start()
t1.join(); t2.join()

### Question 6.3
Look at the timestamps. Did each message arrive at the consumer IMMEDIATELY after the producer sent it? Is there any buffering?

*Your answer:*



## Exercise 6.4 — Design: sbot with Redis

No code — just thinking. Sketch how sbot would work with Redis Streams:

```
Streams:
  sbot:inbound           — all channels push here
  sbot:outbound:telegram — agent pushes, Telegram consumes
  sbot:outbound:cli      — agent pushes, CLI consumes
```

Questions to answer:

**Q1:** How does the agent know which outbound stream to write to? (Hint: the InboundMessage has a `channel` field)

*Your answer:*



**Q2:** What happens if the Telegram process crashes? Do messages get lost?

*Your answer:*



**Q3:** Could you run 3 agent processes reading from the same `sbot:inbound` stream? What feature of Redis Streams would you use? (Hint: consumer groups)

*Your answer:*



## Summary

| Approach | Buffering? | Processes | Complexity |
|----------|-----------|-----------|------------|
| asyncio.Queue (in-process) | Yes — must yield between emits | 1 | Simple |
| Sync callback + sleep(0.05) | Mostly fixed | 1 | Medium |
| Redis Streams (separate processes) | No — producer/consumer are independent | 2+ | Higher |

## Congratulations!

You now understand:
1. Why Python needs `async` (GIL, single-threaded concurrency)
2. How the event loop schedules tasks (cooperative, `await` = yield point)
3. Why blocking code freezes everything (`run_in_executor` fix)
4. Producer/consumer with `asyncio.Queue` (and why producers must yield)
5. Background tasks with `create_task` and signaling with `Event`
6. The exact sbot buffering bug: sync callbacks vs async queues
7. How Redis Streams eliminate the problem entirely

All the async issues in sbot should now make complete sense.